In [1]:
# Al inicio del notebook, agregar estas líneas:
%load_ext autoreload
%autoreload 2

import matplotlib.image as mpimg
import numpy as np
import subprocess
import shutil
import sys
import os
import concurrent.futures

from datetime import datetime
from pathlib import Path

from RRAM import simulation_config

In [2]:
ruta_raiz = os.getcwd() + "/"
print("Ruta raiz del proyecto:", ruta_raiz)
sys.path.append(ruta_raiz)

# Ruta proporcionada
ruta_exp_data = ruta_raiz + "Datos_Experimentales/Ciclos_Experimentales"
ruta_init_data = ruta_raiz + "Initial_data/"

# Función para borrar y crear una carpeta de forma segura
def safe_reset_folder(folder_path):
    # Evita borrar carpetas peligrosas como la raíz del usuario
    if folder_path.strip().lower() in [
        "c:/users/usuario",
        "c:\\users\\usuario",
        "c:/users",
        "c:\\users",
    ]:
        print(f"ADVERTENCIA: No se permite borrar la carpeta protegida: {folder_path}")
        return
    try:
        if os.path.exists(folder_path):
            shutil.rmtree(folder_path)
        os.makedirs(folder_path)
    except PermissionError as e:
        print(
            f"Error de permisos al intentar borrar o crear la carpeta: {folder_path}\n{e}"
        )
    except Exception as e:
        print(f"Error inesperado con la carpeta: {folder_path}\n{e}")

# Borro los datos iniciales de la simulación anterior de forma segura
safe_reset_folder(ruta_init_data)

Ruta raiz del proyecto: c:\Users\Usuario\Documents\GitHub\RRAM_Simulation/


In [3]:
# Solicitar la ruta del archivo al usuario
ruta_archivo_set = ruta_exp_data + '/Cycle_p_1000.txt'
ruta_archivo_reset = ruta_exp_data + '/Cycle_n_1000.txt'

# ruta_archivo_set = ruta_exp_data + "/Mean_DC_Set_1.txt"
# ruta_archivo_reset = ruta_exp_data + "/Mean_DC_Reset_1.txt"

# Leer datos del archivo
data_set = np.loadtxt(ruta_archivo_set, skiprows=1)  # Skip the header row
data_reset = np.loadtxt(ruta_archivo_reset, skiprows=1)  # Skip the header row

ruta_init_script = ruta_raiz + 'Init_simulation.py'
ruta_simulation_script = ruta_raiz + 'RRAM_Simulation_exceptions.py'

# Defino la carpeta donde se guardan los datos iniciales de la simulación
carpeta_results = 'Results'

simulation_path = os.path.join(carpeta_results)

# Verifica si la carpeta existe
if os.path.exists(simulation_path):
    # Elimina la carpeta y su contenido
    shutil.rmtree(simulation_path)

    # crea la carpeta
    os.makedirs(simulation_path)
    os.makedirs(simulation_path + '/Figures')

In [4]:
manager = simulation_config.ConfigManager()
carpeta_init = "Init_data"

# 1. Definimos las listas originales respetando tu "fuerza bruta" de Montecarlo
# Generamos listas repetidas 10 veces para E_a y E_r para replicar la aleatoriedad
factors_set = [1] # Repite el valor 1 diez veces para E_a
factors_reset = [1]

E_a_base = 0.94
E_r_base = 0.955

# Valores de pendiente temperatura para el barrido son 5 valores partiendo de 26 hasta su 90% (aproximadamente 23.4) redondeados a 3 decimales
pendiente_temperatura_values = np.round(np.linspace(-26, -23.4, 5), 3).tolist()

print("Valores de pendiente de temperatura para el barrido:", pendiente_temperatura_values)

# # 2. Diccionario de barrido (Grid Search)
# parametros_barrido = {
#     # Constantes y corrientes (Añadido el nuevo barrido de I_0_reset)
#     "gamma": [10],
#     "I_0_set": [1.8613e-01],
#     "I_0_reset": [1.1221e-01], 
    
#     # Nuevo barrido de Temperatura
#     "init_temp": [300.0],
    
#     # Resistencias (Puedes añadir más valores aquí si quieres)
#     "ohm_resistence_set": [250],
#     "ohm_resistence_reset": [250],
#     "r_termica_no_percola": [10000],
    
#     # Genera de Oxígeno
#     "num_oxigenos_pp_reset_1": [1],
#     "num_oxigenos_pp_reset_2": [3],
#     "num_oxigenos_sp_reset": [5],
    
#     # Repito varias simulaciones con los mismos valores para tener varios casos "iguales"
#     "generation_energy": [round(E_a_base * f, 3) for f in factors_set],
#     "recombination_energy": [round(E_r_base * f, 3) for f in factors_reset],
# }

# 2. Diccionario de barrido (Grid Search)
# Todo lo que cruces aquí se multiplicará matemáticamente
parametros_barrido = {
    # Constantes
    "gamma": [10],
    # Resistencias
    "ohm_resistence_set": [150],
    "ohm_resistence_reset": [150],
    "factor_generar_calor": [2e-6],
    "pendiente_temperatura": [-24.7]*70,  # Repite los valores de pendiente de temperatura para tener más simulaciones
    # Repito varias simulaciones con los mismos valores para tener varios casos "iguales"
    "generation_energy": [round(E_a_base * f, 3) for f in factors_set],
    "recombination_energy": [round(E_r_base * f, 3) for f in factors_reset],
}

# 3. Construimos y exportamos
manager.add_sweep(sweep_params=parametros_barrido)
manager.export_to_init_data(carpeta_init)

# 4. Lanzamos el script de creación de matrices (Init_simulation.py)
# Ya no hace falta pasarle argumentos porque lee el CSV automáticamente
import subprocess
print("\nLlamando a Init_simulation.py...")
resultado = subprocess.run(["python", ruta_init_script], capture_output=True, text=True)
print(resultado.stdout)
if resultado.stderr:
    print("ERRORES:", resultado.stderr)

Valores de pendiente de temperatura para el barrido: [-26.0, -25.35, -24.7, -24.05, -23.4]
-> Generadas 70 combinaciones.
✅ Exportados parámetros y constantes (70 casos) a Init_data/

Llamando a Init_simulation.py...
Construyendo estados iniciales para 70 simulaciones...
Estados iniciales generados correctamente.



In [5]:
# Paralelización con hilos (funciona en Jupyter)
num_simulaciones = len(manager.simulations)

print(f"Se van a ejecutar {num_simulaciones} simulaciones\n")

ruta_raiz = Path.cwd()
carpeta_results = ruta_raiz / "Results"

# Elimino la capreta de resultados anterior
if (carpeta_results).exists():
    shutil.rmtree(carpeta_results)
    carpeta_results.mkdir(parents=True, exist_ok=True)

global_figures_path = Path.cwd() / "Results" / "Figures"
global_figures_path.mkdir(parents=True, exist_ok=True)

Se van a ejecutar 70 simulaciones



In [6]:
import time  # Importamos el módulo para medir el tiempo

path = os.getcwd() + '/'

# Carpeta donde se guardarán los logs
log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)  # Crea la carpeta si no existe

# Limpio la carpeta de logs antes de empezar
for file in os.listdir(log_dir):
    file_path = os.path.join(log_dir, file)
    try:
        if os.path.isfile(file_path):
            os.unlink(file_path)
    except Exception as e:
        print(f"No se pudo eliminar {file_path}. Error: {e}")

# Función que ejecuta una simulación y guarda su propio log
guardar_datos = False  # Cambiado a False para evitar guardar datos de configuracion en cada simulación
num_filametos = 2

def ejecutar_simulacion(num_simulacion):
    log_file_path = os.path.join(log_dir, f"log_simulacion_{num_simulacion + 1}.log")

    with open(log_file_path, 'w') as log_file:
        print(f"Iniciando simulación {num_simulacion+1}")
        subprocess.run(
            [
                "python",
                path + "RRAM_mod_simulation.py",
                str(num_simulacion),
                str(guardar_datos),
                str(num_filametos),
            ],
            stdout=log_file,
            stderr=log_file,
        )

# Inicia el contador de tiempo
start_time = time.time()

print(f"Se van a ejecutar {num_simulaciones} simulaciones:\n")

num_procesadores = int(0.7*(os.cpu_count())) or 10  # type: ignore

with concurrent.futures.ThreadPoolExecutor(
    max_workers=num_procesadores
) as executor:  # Usa hilos en vez de procesos
    executor.map(ejecutar_simulacion, range(num_simulaciones))

# Finaliza el contador de tiempo
end_time = time.time()

# Calcula e imprime el tiempo total de ejecución
elapsed_time = end_time - start_time

elapsed_time=elapsed_time/3600  # Convertir a horas

print(f"Todas las simulaciones han terminado")
print(f"Tiempo total de ejecución: {elapsed_time:.2f} horas")

Se van a ejecutar 70 simulaciones:

Iniciando simulación 1
Iniciando simulación 2
Iniciando simulación 3
Iniciando simulación 4
Iniciando simulación 5
Iniciando simulación 6
Iniciando simulación 7
Iniciando simulación 8
Iniciando simulación 9
Iniciando simulación 10
Iniciando simulación 11
Iniciando simulación 12
Iniciando simulación 14
Iniciando simulación 13
Iniciando simulación 15
Iniciando simulación 16
Iniciando simulación 17
Iniciando simulación 18
Iniciando simulación 19
Iniciando simulación 20
Iniciando simulación 21
Iniciando simulación 22
Iniciando simulación 23
Iniciando simulación 24
Iniciando simulación 25
Iniciando simulación 26
Iniciando simulación 27
Iniciando simulación 28
Iniciando simulación 29
Iniciando simulación 30
Iniciando simulación 31
Iniciando simulación 32
Iniciando simulación 33
Iniciando simulación 34
Iniciando simulación 35
Iniciando simulación 36
Iniciando simulación 37
Iniciando simulación 38
Iniciando simulación 39
Iniciando simulación 40
Iniciando sim

In [7]:
# # Copio al backup los resultados y logs de la simulación actual con hora y dia
# backup_path = "C:/Users/Usuario/Documents/Backup"

# # Obtener fecha y hora actual
# ahora = datetime.now()

# # Formatear con strftime
# formato = "%d-%m_%Y__%H_%M_%S"
# fecha_formateada = ahora.strftime(formato)

# # creo la carpeta de destino en el backup
# backup_path_results = Path(backup_path) / f"Resultados_{fecha_formateada}"
# backup_path_logs = Path(backup_path) / f"Logs_{fecha_formateada}"

# source_path_results = Path.cwd() / "Results"
# source_path_logs = Path.cwd() / "Logs"

# destination_results_path = Path(backup_path_results)
# destination_logs_path = Path(backup_path_logs)


# # Copiar todo el directorio de la simulación al backup
# shutil.copytree(source_path_results, destination_results_path)
# shutil.copytree(source_path_logs, destination_logs_path)

# print(f"Datos de la simulación guardados en el backup.")

In [8]:
# import nbformat
# from nbconvert.preprocessors import ExecutePreprocessor

# def ejecutar_notebook(ruta_notebook, timeout=600):
#     # Cargar el notebook
#     with open(ruta_notebook, 'r', encoding='utf-8') as f:
#         notebook = nbformat.read(f, as_version=4)
    
#     # Configurar el preprocesador
#     ep = ExecutePreprocessor(timeout=timeout, kernel_name='python3')
    
#     # Ejecutar el notebook
#     ep.preprocess(notebook, {'metadata': {'path': './'}})

# # Ejemplo de uso
# ejecutar_notebook('Mejor_Curva_IV_exp.ipynb')